# NB05: Build Reference Metabolic Models via KBUtilLib

Build genome-scale metabolic models for reference organisms spanning
Gram-negative, Gram-positive, and Archaea using the KBUtilLib pipeline:
genome classification -> template selection -> MSBuilder -> ATP correction -> gapfilling.

These models will be used in NB06-08 to evaluate the impact of
thermodynamic updates.

In [ ]:
import sys
import os
import json
import pickle
from pathlib import Path

import cobra
import pandas as pd

KBUTILLIB_ROOT = Path('/global_share/KBaseUtilities/KBUtilLib')
sys.path.insert(0, str(KBUTILLIB_ROOT / 'src'))

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

from kbutillib import MSReconstructionUtils, MSFBAUtils

## 1. Initialize reconstruction utilities and genome classifier

In [ ]:
recon_utils = MSReconstructionUtils()

from modelseedpy.helpers import get_template
from modelseedpy.core.msgenomeclassifier import MSGenomeClassifier

genome_classifier = MSGenomeClassifier()
print('Reconstruction utilities and genome classifier initialized')

## 2. Load reference genomes from KBase

We need genomes for:
- **E. coli K-12 MG1655** (Gram-negative) — best-characterized model organism
- **B. subtilis 168** (Gram-positive) — classic Gram-positive reference
- **P. putida KT2440** (Gram-negative) — environmentally diverse metabolism

Genomes are loaded from KBase workspace or from local RAST annotations.

In [ ]:
from cobrakbase.core.kbasegenome import KBaseGenome
from modelseedpy.core.msgenome import MSGenome

# Define reference organisms
# Genome refs should be updated to point to actual KBase workspace objects
reference_organisms = {
    'ecoli': {
        'name': 'Escherichia coli K-12 MG1655',
        'template': 'gn',
        'genome_ref': None,  # Fill with KBase workspace ref
    },
    'bsubtilis': {
        'name': 'Bacillus subtilis 168',
        'template': 'gp',
        'genome_ref': None,  # Fill with KBase workspace ref
    },
    'pputida': {
        'name': 'Pseudomonas putida KT2440',
        'template': 'gn',
        'genome_ref': None,  # Fill with KBase workspace ref
    },
}

print('Reference organisms defined. Genome refs need to be filled with KBase workspace references.')
print('Update the genome_ref values above with actual workspace object references.')

## 3. Build models

For each genome, run the full KBUtilLib reconstruction pipeline.

In [ ]:
models = {}
build_results = []

for org_id, org_info in reference_organisms.items():
    print(f'\n=== Building model for {org_info["name"]} ===')

    genome = org_info.get('genome_obj')
    if genome is None:
        print(f'  Skipping {org_id} — no genome object loaded')
        continue

    output, mdlutl = recon_utils.build_metabolic_model(
        genome=genome,
        genome_classifier=genome_classifier,
        gs_template=org_info['template'],
        atp_safe=True,
        max_gapfilling=10,
    )

    if mdlutl is None:
        print(f'  Model building failed: {output.get("Comments", [])}')
        continue

    models[org_id] = mdlutl
    build_results.append({
        'organism': org_id,
        'name': org_info['name'],
        'class': output.get('Class', ''),
        'genes': output.get('Genes', 0),
        'model_genes': output.get('Model genes', 0),
        'reactions': output.get('Reactions', 0),
        'core_gf': output.get('Core GF', 'NA'),
    })
    print(f'  Reactions: {output["Reactions"]}, Genes: {output["Model genes"]}')

if build_results:
    df_build = pd.DataFrame(build_results)
    print('\nBuild summary:')
    print(df_build.to_string(index=False))

## 4. Test basic growth

In [ ]:
fba_utils = MSFBAUtils()

for org_id, mdlutl in models.items():
    solution = fba_utils.run_fba(mdlutl)
    print(f'{org_id}: growth = {solution.objective_value:.4f}')

## 5. Save models

In [ ]:
for org_id, mdlutl in models.items():
    model_path = DATA_DIR / f'model_{org_id}.json'
    cobra.io.save_json_model(mdlutl.model, str(model_path))
    print(f'Saved {org_id} model to {model_path}')

if build_results:
    df_build.to_csv(DATA_DIR / 'model_build_summary.tsv', sep='\t', index=False)
    print(f'Saved build summary to data/model_build_summary.tsv')